# Build 1 — operational schema validation (Lakebase Postgres)

**Execution proof** that the modeled operational schema in `operational_schema.sql` actually ran on the
Lakebase Postgres **production** branch — the related tables, primary keys, and foreign keys exist in the
live database. This notebook connects to Lakebase Postgres
(`brightwave-campaign-desk` / `brightwave_lakebase_tkop`, schema `app`) and reads `information_schema` /
`pg_catalog`. Export it **with cell outputs** as the committed evidence.

Domain model (Campaign Desk):
```
conversations 1──* messages 1──* feedback              (chat / assist domain)
campaign_actions_app 1──* workflow_events              (action / decision domain)
creatives_search                                       (searchable creative catalog)
```
The credential is minted at runtime from the notebook's own identity — no secret is written into this file.


In [1]:
%pip install --quiet psycopg2-binary


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [2]:
import psycopg2, pandas as pd
from databricks.sdk import WorkspaceClient

HOST     = "ep-delicate-mountain-d8kizg3e.database.us-east-2.cloud.databricks.com"
DBNAME   = "brightwave_lakebase_tkop"
SCHEMA   = "app"
ENDPOINT = "projects/brightwave-campaign-desk/branches/production/endpoints/primary"

# Mint a short-lived Lakebase OAuth credential as THIS notebook's identity (never printed / committed).
w = WorkspaceClient()
cred = w.api_client.do("POST", "/api/2.0/postgres/credentials", body={"endpoint": ENDPOINT})
token = cred["token"]

conn = psycopg2.connect(host=HOST, dbname=DBNAME, user=w.current_user.me().user_name,
                        password=token, sslmode="require")
conn.set_session(readonly=True)

def q(sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

who = q("SELECT current_user, current_database(), version()")
print("Connected as :", who.iloc[0]["current_user"])
print("Database     :", who.iloc[0]["current_database"])
print("Server       :", who.iloc[0]["version"][:60], "...")


Connected as : theodore.kop@databricks.com
Database     : brightwave_lakebase_tkop
Server       : PostgreSQL 17.11 (32e7196) on x86_64-pc-linux-gnu, compiled  ...


## 1. The modeled tables exist in schema `app` (the DDL ran)


In [3]:
tables = q(f"""
  SELECT table_name,
         (SELECT count(*) FROM information_schema.columns c
           WHERE c.table_schema=t.table_schema AND c.table_name=t.table_name) AS n_columns
    FROM information_schema.tables t
   WHERE table_schema='{SCHEMA}' AND table_type='BASE TABLE'
   ORDER BY table_name
""")
print(tables.to_string(index=False))
print(f"\n{len(tables)} tables in schema '{SCHEMA}'")


          table_name  n_columns
campaign_actions_app         12
       conversations          6
    creatives_search          8
            feedback          8
            messages         10
     workflow_events          8

6 tables in schema 'app'


## 2. Primary keys — every table is keyed


In [4]:
pks = q(f"""
  SELECT tc.table_name, kcu.column_name AS pk_column
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu
      ON tc.constraint_name=kcu.constraint_name AND tc.table_schema=kcu.table_schema
   WHERE tc.table_schema='{SCHEMA}' AND tc.constraint_type='PRIMARY KEY'
   ORDER BY tc.table_name
""")
print(pks.to_string(index=False))


          table_name   pk_column
campaign_actions_app          id
       conversations          id
    creatives_search creative_id
            feedback          id
            messages          id
     workflow_events          id


## 3. Foreign keys — the RELATED tables (this is the domain ER model)


In [5]:
fks = q(f"""
  SELECT tc.table_name        AS child_table,
         kcu.column_name       AS fk_column,
         ccu.table_name        AS parent_table,
         ccu.column_name       AS parent_column,
         tc.constraint_name
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu
      ON tc.constraint_name=kcu.constraint_name AND tc.table_schema=kcu.table_schema
    JOIN information_schema.constraint_column_usage ccu
      ON tc.constraint_name=ccu.constraint_name AND tc.table_schema=ccu.table_schema
   WHERE tc.table_schema='{SCHEMA}' AND tc.constraint_type='FOREIGN KEY'
   ORDER BY tc.table_name
""")
print(fks.to_string(index=False))
print("\nRelationships:")
for _, r in fks.iterrows():
    print(f"  {r['child_table']}.{r['fk_column']}  ->  {r['parent_table']}.{r['parent_column']}")


    child_table       fk_column         parent_table parent_column                              constraint_name
       feedback      message_id             messages            id           feedback_message_id_messages_id_fk
       messages conversation_id        conversations            id messages_conversation_id_conversations_id_fk
workflow_events       action_id campaign_actions_app            id                    fk_workflow_events_action

Relationships:
  feedback.message_id  ->  messages.id
  messages.conversation_id  ->  conversations.id
  workflow_events.action_id  ->  campaign_actions_app.id


## 4. Column model per table (typed domain columns)


In [6]:
cols = q(f"""
  SELECT table_name, ordinal_position AS pos, column_name, data_type, is_nullable
    FROM information_schema.columns
   WHERE table_schema='{SCHEMA}'
   ORDER BY table_name, ordinal_position
""")
for t in tables["table_name"]:
    sub = cols[cols["table_name"] == t][["pos", "column_name", "data_type", "is_nullable"]]
    print(f"\n=== app.{t} ===")
    print(sub.to_string(index=False))



=== app.campaign_actions_app ===
 pos         column_name                data_type is_nullable
   1                  id                     uuid          NO
   2         campaign_id                     text          NO
   3         action_type                     text          NO
   4  target_campaign_id                     text         YES
   5       drafted_brief                     text         YES
   6 predicted_roas_lift         double precision         YES
   7              status                     text          NO
   8         approved_by                     text         YES
   9         audit_trail                    jsonb          NO
  10          created_at timestamp with time zone          NO
  11          decided_at timestamp with time zone         YES
  12      priority_score         double precision         YES

=== app.conversations ===
 pos column_name                data_type is_nullable
   1          id                     uuid          NO
   2  user_email         

## 5. Live row counts (the schema is an operational, populated database)


In [7]:
counts = []
for t in tables["table_name"]:
    n = q(f"SELECT COUNT(*) AS n FROM {SCHEMA}.{t}").iloc[0]["n"]
    counts.append({"table": f"{SCHEMA}.{t}", "rows": int(n)})
print(pd.DataFrame(counts).to_string(index=False))


                   table  rows
app.campaign_actions_app     6
       app.conversations    11
    app.creatives_search   400
            app.feedback     1
            app.messages    48
     app.workflow_events     5


---
**Result:** all 6 modeled tables exist with primary keys, and the 3 foreign keys wire the domain together
(`feedback → messages → conversations`, `workflow_events → campaign_actions_app`). The schema in
`operational_schema.sql` demonstrably ran on the live Lakebase Postgres database — these cell outputs are the
committed proof-of-execution.


In [8]:
conn.close()
